# Random Forest Baseline for ICBHI Respiratory Sound Classification

This notebook builds a traditional machine-learning baseline for respiratory-cycle classification using the ICBHI 2017 dataset.

The workflow is to load metadata and audio, extract log-Mel features, train a Random Forest on four respiratory categories, then evaluate **Normal vs. Abnormal** classification using sensitivity, specificity, and the harmonic score (HS).

**Hint:** Firstly, run ` 01.data_analysis_&_feature_extraction.ipynb ` to know the data better and see the visualization charts, then run this notebook from top to bottom and verify that dataset paths match your local project structure.

In [1]:
import os
import numpy as np
import pandas as pd

import librosa
import matplotlib.pyplot as plt

import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score

### Load Metadata and Set the Audio Directory

The ` split_metadata.csv` dataset provides the source file, cycle start/end times, category, and assigned officail data split.<br>
`AUDIO_DIR` points to the WAV audio files.

**Hint:** Paths are relative to the notebook's current working directory. If a file is not found, inspect `Path.cwd()` and update the paths.

In [10]:
df = pd.read_csv("..//datasets//split_metadata.csv")
AUDIO_DIR = "../datasets/audios/"

### Extract Log-Mel Spectrograms

`extract_logmel()` converts a respiratory-cycle waveform into a log-Mel spectrogram using a 16 kHz sample rate, 32 Mel bands, and a frequency range of 50–2,000 Hz.

**Hint:** Keep these feature-extraction settings consistent when reusing this pipeline or comparing experiments.

In [3]:
SR = 16000
N_MELS = 32

def extract_logmel(y, sr=SR):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=1024,
        hop_length=256,
        win_length=1024,
        n_mels=N_MELS,
        fmin=50,
        fmax=2000,
        power=2.0
    )
    log_mel = librosa.power_to_db(mel, ref=1.0)
    return log_mel


In [4]:
def logmel_to_features(log_mel):
    """
    log_mel: (n_mels, time)
    returns: (4 * n_mels,)
    """
    feats = []
    
    feats.extend(np.mean(log_mel, axis=1))
    feats.extend(np.std(log_mel, axis=1))
    feats.extend(np.min(log_mel, axis=1))
    feats.extend(np.max(log_mel, axis=1))

    return np.array(feats)


### Map Respiratory Categories to Numeric Labels

The four labels are `0 = Healthy`, `1 = Only Crackle`, `2 = Only Wheeze`, and `3 = Crackle and Wheeze`. These labels are used for multiclass training; the later binary evaluation groups labels 1–3 as abnormal.


In [5]:
def icbhi_label(category):
    if category == 'Healthy':
        return 0
    elif category == 'Only Crackle':
        return 1  
    elif category == 'Only Wheeze':
        return 2
    else:
        return 3  #crackle & wheeze (both)


In [11]:
X = []
y = []
splits = []

for idx, row in df.iterrows():

    file_path = os.path.join(AUDIO_DIR, row["source_file"] + ".wav")
    signal, sr = librosa.load(file_path, sr=SR)

    start_sample = int(row["start"] * sr)
    end_sample = int(row["end"] * sr)

    if end_sample <= start_sample:
        continue

    cycle = signal[start_sample:end_sample]

    log_mel = extract_logmel(cycle)
    features = logmel_to_features(log_mel)

    label = icbhi_label(row["label"])

    X.append(features)
    y.append(label)
    splits.append(row["split"])

    print(
        f"\rProcessing {idx} - {row['source_file']} | label={label} | split={row['split']}",
        end="",
        flush=True
    )


X = np.array(X)
y = np.array(y)
splits = np.array(splits)
print("\n")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("splits shape:", splits.shape)

Processing 6897 - 226_1b1_Pl_sc_LittC2SE | label=0 | split=train

X shape: (6898, 128)
y shape: (6898,)
splits shape: (6898,)


In [13]:
df["label"] = df["label"].apply(icbhi_label)
df["label"].value_counts()


label
0    3642
1    1864
2     886
3     506
Name: count, dtype: int64

In [14]:
X_train = X[splits == "train"]
y_train = y[splits == "train"]
X_test  = X[splits == "test"]
y_test  = y[splits == "test"]

print("Train samples:", X_train.shape)
print("Test samples:", X_test.shape)

print("Train label distribution:", np.bincount(y_train))
print("Test label distribution:", np.bincount(y_test))

Train samples: (4142, 128)
Test samples: (2756, 128)
Train label distribution: [2063 1215  501  363]
Test label distribution: [1579  649  385  143]


### Train the Random Forest

The classifier uses 200 trees, a fixed random seed for reproducibility, and all available CPU cores (`n_jobs=-1`). It is trained to predict the four respiratory categories.


In [15]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
y_pred = rf.predict(X_test)

### Convert Four-Class Predictions to Binary Outcomes

For binary evaluation, label `0` is **Normal** and labels `1–3` are **Abnormal**. The confusion matrix provides TN, FP, FN, and TP.

**Hint:** These metrics evaluate normal-versus-abnormal detection, not the distinction between crackles and wheezes.

In [17]:
y_test_binary = (y_test != 0).astype(int)
y_pred_binary = (y_pred != 0).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_binary,
    y_pred_binary
).ravel()

In [18]:
SE = tp / (tp + fn)
SP = tn / (tn + fp)

HS = (2 * SE * SP) / (SE + SP)

print(f"Sensitivity (SE): {SE:.4f}")
print(f"Specificity (SP): {SP:.4f}")
print(f"Harmonic Score (HS): {HS:.4f}")

Sensitivity (SE): 0.4078
Specificity (SP): 0.7042
Harmonic Score (HS): 0.5165


In [21]:
metadata = {
    "model": "RandomForestClassifier",
    "features": "log-mel statistics (mean/std/min/max)",
    "n_mels": N_MELS,
    "sampling_rate": SR,
    "labels": {
        0: "Normal",
        1: "Crackles",
        2: "Wheezes",
        3: "Both"
    },
    "dataset": "ICBHI 2017",
    "split": "Official patient-wise 60/40"
}

output_dir = os.path.join("../models", "random_forest_baseline")
os.makedirs(output_dir, exist_ok=True)
joblib.dump(metadata, os.path.join(output_dir, "rf_logmel_icbhi_metadata.joblib"))

['../models\\random_forest_baseline\\rf_logmel_icbhi_metadata.joblib']